# Advanced Tutorial Problems with Solutions: Copying Python Sets

This is a **new tutorial-style notebook** on the same topic: copying sets in Python.

Instead of presenting large solution blocks immediately, each problem is broken into small logical steps:

1. establish the setup;
2. make a prediction;
3. run one focused experiment;
4. interpret the result;
5. complete the solution with assertions.

The notebook focuses on advanced reasoning about:

- assignment versus copying;
- shallow copies;
- deep copies;
- object identity;
- mutable set elements;
- hash stability;
- equality/hash contracts;
- nested aliasing;
- shared object graphs;
- cycles;
- custom `__deepcopy__`;
- set subclasses;
- defensive copying;
- immutable snapshots;
- testing copy semantics.


## Best-practice rule used throughout

Sets are unordered.

We will therefore avoid solutions that depend on display or iteration order. When we need a specific object, we will identify it by a stable attribute such as `name`, `id`, or another key.


In [2]:
from copy import copy, deepcopy
from dataclasses import dataclass


# Part 1 — Assignment is not copying


## Problem 1 — Two names, one set

Consider:

```python
first = {"A", "B"}
second = first
second.add("C")
```

Before running the next cell, answer:

- Was a second set created?
- What does `first is second` return?
- Does `"C"` appear in `first`?


In [3]:
first = {"A", "B"}
second = first

print("same object:", first is second)

second.add("C")

print("first:", first)
print("second:", second)


same object: True
first: {'B', 'A', 'C'}
second: {'B', 'A', 'C'}


### Solution 1

No copy was made.

`second = first` creates another reference to the same set object. Therefore a mutation performed through either name is visible through both names.


In [4]:
assert first is second
assert first == {"A", "B", "C"}
assert second == {"A", "B", "C"}


### Mental model

Keep two levels separate:

- **outer set identity** — is it the same set object?
- **element identity** — do two sets contain references to the same member objects?

Shallow copies change the first level but not normally the second.


# Part 2 — A mutable object inside a set


To expose shallow-copy behavior clearly, we need mutable objects that are also hashable.

A simple user-defined class is normally hashable by identity unless we override equality/hash behavior.


In [5]:
class Note:
    def __init__(self, title):
        self.title = title

    def __repr__(self):
        return f"Note(title={self.title!r})"


n1 = Note("Sets")
n2 = Note("Dictionaries")

notes = {n1, n2}


## Problem 2 — Why is this legal?

`Note` instances are mutable because `title` can change.

Yet this works:

```python
notes = {n1, n2}
```

Why?


In [6]:
print("hash(n1):", hash(n1))
print("hash(n2):", hash(n2))


hash(n1): 78606233794
hash(n2): 78606105461


### Solution 2

Set elements need to be **hashable**.

A normal user-defined object is typically hashable using identity-based behavior. Mutability by itself does not automatically make a custom object unhashable.


# Part 3 — Three shallow-copy techniques


For a normal set, common shallow-copy forms include:

```python
a = s.copy()
b = set(s)
c = {*s}
```

We will verify both container identity and member identity.


## Problem 3 — Are the outer sets new?


In [7]:
by_method = notes.copy()
by_constructor = set(notes)
by_unpacking = {*notes}

print(by_method is notes)
print(by_constructor is notes)
print(by_unpacking is notes)


False
False
False


### Step 1 result

All three are new outer set objects.


In [8]:
assert by_method is not notes
assert by_constructor is not notes
assert by_unpacking is not notes


### Step 2 — Are the contained objects new?

Because sets are unordered, compare sets of object IDs.


In [9]:
def element_ids(s):
    return {id(item) for item in s}


print("original:", element_ids(notes))
print("method:", element_ids(by_method))
print("constructor:", element_ids(by_constructor))
print("unpacking:", element_ids(by_unpacking))


original: {1257699740704, 1257697687376}
method: {1257699740704, 1257697687376}
constructor: {1257699740704, 1257697687376}
unpacking: {1257699740704, 1257697687376}


### Solution 3

All direct element identities are shared.

That is the central shallow-copy rule:

> New outer set, same referenced member objects.


In [10]:
assert element_ids(notes) == element_ids(by_method)
assert element_ids(notes) == element_ids(by_constructor)
assert element_ids(notes) == element_ids(by_unpacking)


# Part 4 — Membership mutation versus element mutation


A shallow copy separates set membership.

That means adding or removing elements from one set does not modify the other set's membership.

But both sets may still point to the same mutable objects.


## Problem 4 — Two kinds of mutation

We will first mutate the set itself.


In [11]:
original = {n1, n2}
shallow = original.copy()

n3 = Note("Functions")
shallow.add(n3)

print("n3 in original:", n3 in original)
print("n3 in shallow:", n3 in shallow)


n3 in original: False
n3 in shallow: True


### Step 1 interpretation

Adding `n3` changes only `shallow`.

The outer set structures are independent.


In [12]:
assert n3 not in original
assert n3 in shallow


### Step 2 — Now mutate a shared member


In [13]:
n1.title = "Advanced Sets"

print("original:", original)
print("shallow:", shallow)


original: {Note(title='Advanced Sets'), Note(title='Dictionaries')}
shallow: {Note(title='Functions'), Note(title='Advanced Sets'), Note(title='Dictionaries')}


### Solution 4

The title change is visible through both sets because both contain the same `n1` object.

A useful summary is:

> Shallow copying isolates membership, not member state.


In [14]:
assert any(note.title == "Advanced Sets" for note in original)
assert any(note.title == "Advanced Sets" for note in shallow)


# Part 5 — `copy.copy()` and sets


The `copy` module provides a general shallow-copy function:

```python
copy(obj)
```

For a normal set, it behaves as a shallow copy.


## Problem 5 — Compare `copy()` and `.copy()`


In [15]:
base = {Note("A"), Note("B")}

using_method = base.copy()
using_module = copy(base)

print("method shares outer identity:", using_method is base)
print("module shares outer identity:", using_module is base)

print(
    "method shares element identities:",
    element_ids(using_method) == element_ids(base),
)

print(
    "module shares element identities:",
    element_ids(using_module) == element_ids(base),
)


method shares outer identity: False
module shares outer identity: False
method shares element identities: True
module shares element identities: True


### Solution 5

Both create new outer sets while reusing the same direct member objects.

For a known set, `.copy()` is usually the clearest expression of intent.


In [16]:
assert using_method is not base
assert using_module is not base
assert element_ids(using_method) == element_ids(base)
assert element_ids(using_module) == element_ids(base)


# Part 6 — First deep-copy experiment


A deep copy recursively copies many mutable objects reachable from the original object.

We will use objects that themselves contain mutable lists.


In [17]:
class Course:
    def __init__(self, name, topics):
        self.name = name
        self.topics = list(topics)

    def __repr__(self):
        return f"Course({self.name!r}, topics={self.topics!r})"


python_course = Course("Python", ["sets", "dicts"])
db_course = Course("Databases", ["sql", "indexes"])

courses = {python_course, db_course}


## Problem 6 — What does `deepcopy()` duplicate?

First make the copy and inspect only direct identities.


In [18]:
courses_copy = deepcopy(courses)

print("same outer set:", courses_copy is courses)
print(
    "shared direct member identities:",
    element_ids(courses) & element_ids(courses_copy),
)


same outer set: False
shared direct member identities: set()


### Step 1 result

The outer set is new, and the direct member objects are also new.


In [19]:
assert courses_copy is not courses
assert element_ids(courses).isdisjoint(element_ids(courses_copy))


### Step 2 — Find corresponding objects without relying on set order


In [20]:
original_python = next(
    course for course in courses
    if course.name == "Python"
)

copied_python = next(
    course for course in courses_copy
    if course.name == "Python"
)

print(original_python)
print(copied_python)


Course('Python', topics=['sets', 'dicts'])
Course('Python', topics=['sets', 'dicts'])


### Step 3 — Compare nested list identity


In [21]:
print(
    "same topics list:",
    original_python.topics is copied_python.topics,
)


same topics list: False


### Step 4 — Mutate the original nested list


In [22]:
original_python.topics.append("copying")

print("original:", original_python.topics)
print("deep copy:", copied_python.topics)


original: ['sets', 'dicts', 'copying']
deep copy: ['sets', 'dicts']


### Solution 6

The nested list is independent.

This is a real ownership difference from a shallow copy.


In [23]:
assert original_python.topics == ["sets", "dicts", "copying"]
assert copied_python.topics == ["sets", "dicts"]


# Part 7 — Deep copy preserves intentional sharing inside the graph


A deep copy does not simply make every reference point to a unique new object.

It usually preserves alias relationships inside the copied graph.


In [24]:
class Service:
    def __init__(self, name, config):
        self.name = name
        self.config = config

    def __repr__(self):
        return f"Service({self.name!r})"


shared_config = {
    "region": "eu",
    "retries": 3,
}

api = Service("api", shared_config)
worker = Service("worker", shared_config)

services = {api, worker}


## Problem 7 — One shared configuration

First verify that both original services reference the exact same dictionary.


In [25]:
assert api.config is worker.config
print(api.config is worker.config)


True


### Step 2 — Deep-copy the set


In [26]:
services_copy = deepcopy(services)

copied_by_name = {
    service.name: service
    for service in services_copy
}

copied_api = copied_by_name["api"]
copied_worker = copied_by_name["worker"]


### Step 3 — Ask two different identity questions

Does the copied API service share its copied config with the copied worker?

And is that config still the original config?


In [27]:
print(
    "copied services share config:",
    copied_api.config is copied_worker.config,
)

print(
    "copied config is original config:",
    copied_api.config is shared_config,
)


copied services share config: True
copied config is original config: False


### Solution 7

The copied services share **one copied configuration**.

That copied configuration is distinct from the original one.

`deepcopy()` uses internal memoization to preserve these relationships.


In [28]:
assert copied_api.config is copied_worker.config
assert copied_api.config is not shared_config


# Part 8 — Hash stability is a set invariant


This is one of the most important advanced topics.

A set assumes that an element's hash does not change while that element remains stored in the set.


In [29]:
class MutableKey:
    def __init__(self, key):
        self.key = key

    def __eq__(self, other):
        if not isinstance(other, MutableKey):
            return NotImplemented
        return self.key == other.key

    def __hash__(self):
        return hash(self.key)

    def __repr__(self):
        return f"MutableKey({self.key!r})" 


## Problem 8 — Break the hash invariant

Insert an object, then mutate the exact field used by `__hash__`.


In [30]:
item = MutableKey("A")
items = {item}

old_hash = hash(item)

assert item in items

item.key = "B"

new_hash = hash(item)

print("old hash:", old_hash)
print("new hash:", new_hash)
print("hash changed:", old_hash != new_hash)


old hash: -53261520554030773
new hash: 9212578708899603979
hash changed: True


### Step 2 — Ask the set to locate the same object


In [31]:
print("membership test:", item in items)
print("set contents:", items)


membership test: False
set contents: {MutableKey('B')}


### Solution 8

The object may still be physically stored in the set, but lookup can fail because the set originally placed the object according to its old hash.

Best practice:

> Never mutate fields that participate in hashing while the object is inside a set or used as a dictionary key.


# Part 9 — Safe mutable entities


A mutable object can still be safely hashable if equality and hashing depend only on a stable immutable identity field.


In [32]:
class Customer:
    def __init__(self, customer_id, display_name):
        self.customer_id = customer_id
        self.display_name = display_name

    def __eq__(self, other):
        if not isinstance(other, Customer):
            return NotImplemented
        return self.customer_id == other.customer_id

    def __hash__(self):
        return hash(self.customer_id)

    def __repr__(self):
        return (
            f"Customer({self.customer_id!r}, "
            f"display_name={self.display_name!r})"
        )


## Problem 9 — Mutate non-key state safely


In [33]:
customer = Customer("C-001", "Alice")
customers = {customer}

before = hash(customer)

customer.display_name = "Alice Cooper"

after = hash(customer)

print("hash stable:", before == after)
print("membership stable:", customer in customers)


hash stable: True
membership stable: True


### Solution 9

`display_name` is mutable, but it does not participate in equality or hashing.

The stable `customer_id` defines logical identity.


In [34]:
assert before == after
assert customer in customers


# Part 10 — Equality and hash must agree


For hashable objects, Python requires:

> If `a == b`, then `hash(a) == hash(b)`.

The reverse is not required: unequal objects may collide and have the same hash.


## Problem 10 — A broken equality/hash contract


In [35]:
class BadUser:
    def __init__(self, email):
        self.email = email

    def __eq__(self, other):
        if not isinstance(other, BadUser):
            return NotImplemented
        return self.email == other.email

    def __hash__(self):
        return id(self)


u1 = BadUser("same@example.com")
u2 = BadUser("same@example.com")

print("equal:", u1 == u2)
print("hashes:", hash(u1), hash(u2))


equal: True
hashes: 1257699743056 1257700116112


### Why is this broken?

The objects compare equal by email, but their hashes are based on different identities.

That violates Python's hashing contract.


### Solution 10 — Hash the same stable equality key


In [36]:
class GoodUser:
    def __init__(self, email):
        self.email = email

    def __eq__(self, other):
        if not isinstance(other, GoodUser):
            return NotImplemented
        return self.email == other.email

    def __hash__(self):
        return hash(self.email)


u1 = GoodUser("same@example.com")
u2 = GoodUser("same@example.com")

assert u1 == u2
assert hash(u1) == hash(u2)

users = {u1, u2}

print("logical set size:", len(users))


logical set size: 1


# Part 11 — Immutable value objects


When an object's identity is entirely value-based, immutability is often the easiest design.

Frozen dataclasses are useful here.


In [37]:
@dataclass(frozen=True)
class Permission:
    resource: str
    action: str


read_users = Permission("users", "read")
write_users = Permission("users", "write")

permissions = {read_users, write_users}


## Problem 11 — Why is a frozen value object convenient?

Check:

- value equality;
- stable hashing;
- blocked field mutation.


In [38]:
same_read = Permission("users", "read")

print("equal:", same_read == read_users)
print("same hash:", hash(same_read) == hash(read_users))
print("membership:", same_read in permissions)


equal: True
same hash: True
membership: True


In [39]:
try:
    read_users.action = "delete"
except Exception as exc:
    print(type(exc).__name__, exc)


FrozenInstanceError cannot assign to field 'action'


### Solution 11

The frozen dataclass naturally matches set requirements for immutable value-like elements.


# Part 12 — A deep copy may compare unequal


Deep copy and equality are separate concepts.

A class that does not define value equality normally compares distinct instances by identity.


## Problem 12 — Identity-based element equality


In [40]:
a = Note("Python")
b = Note("Python")

print("a == b:", a == b)
print("a is b:", a is b)


a == b: False
a is b: False


### Step 2 — Deep-copy a set of such objects


In [41]:
note_set = {
    Note("Python"),
    Note("SQL"),
}

note_set_copy = deepcopy(note_set)

print("sets equal:", note_set == note_set_copy)
print(
    "direct identities shared:",
    bool(element_ids(note_set) & element_ids(note_set_copy)),
)


sets equal: False
direct identities shared: False


### Solution 12

The copied graph can be correct even if the two sets are not `==`.

The element class has identity-based equality semantics.


# Part 13 — Value equality changes the comparison


In [42]:
class Book:
    def __init__(self, isbn, title):
        self.isbn = isbn
        self.title = title

    def __eq__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return self.isbn == other.isbn

    def __hash__(self):
        return hash(self.isbn)

    def __repr__(self):
        return f"Book({self.isbn!r}, {self.title!r})"


books = {
    Book("ISBN-1", "Python"),
    Book("ISBN-2", "Data"),
}

books_copy = deepcopy(books)


## Problem 13 — Can two sets be equal while sharing no element identities?


In [43]:
print("sets equal:", books == books_copy)
print(
    "shared direct ids:",
    element_ids(books) & element_ids(books_copy),
)


sets equal: True
shared direct ids: set()


### Solution 13

Yes.

Value equality and object identity answer different questions.


In [44]:
assert books == books_copy
assert element_ids(books).isdisjoint(element_ids(books_copy))


# Part 14 — Nested aliasing in shallow copies


In [45]:
class Pipeline:
    def __init__(self, name, stages):
        self.name = name
        self.stages = stages

    def __repr__(self):
        return f"Pipeline({self.name!r})"


pipeline = Pipeline(
    "etl",
    [
        {"name": "extract", "enabled": True},
        {"name": "load", "enabled": True},
    ],
)

pipelines = {pipeline}
pipelines_shallow = pipelines.copy()
pipelines_deep = deepcopy(pipelines)


## Problem 14 — Trace aliasing through several levels

First compare the direct pipeline objects.


In [46]:
shallow_pipeline = next(iter(pipelines_shallow))
deep_pipeline = next(iter(pipelines_deep))

print("shallow shares pipeline:", shallow_pipeline is pipeline)
print("deep shares pipeline:", deep_pipeline is pipeline)


shallow shares pipeline: True
deep shares pipeline: False


### Step 2 — Compare the nested stage lists


In [47]:
print(
    "shallow shares stages list:",
    shallow_pipeline.stages is pipeline.stages,
)

print(
    "deep shares stages list:",
    deep_pipeline.stages is pipeline.stages,
)


shallow shares stages list: True
deep shares stages list: False


### Step 3 — Mutate a dictionary inside the list


In [48]:
pipeline.stages[0]["enabled"] = False

print("original:", pipeline.stages)
print("shallow:", shallow_pipeline.stages)
print("deep:", deep_pipeline.stages)


original: [{'name': 'extract', 'enabled': False}, {'name': 'load', 'enabled': True}]
shallow: [{'name': 'extract', 'enabled': False}, {'name': 'load', 'enabled': True}]
deep: [{'name': 'extract', 'enabled': True}, {'name': 'load', 'enabled': True}]


### Solution 14

The shallow copy shares the top-level `Pipeline`, so it also observes all nested state reachable through that shared object.

The deep copy contains independent nested state here.


# Part 15 — Cyclic object graphs


A cyclic graph can refer back to itself.

Naive recursive copying could recurse forever. `deepcopy()` handles many cycles using memoization.


In [49]:
class Node:
    def __init__(self, name):
        self.name = name
        self.next = None

    def __repr__(self):
        return f"Node({self.name!r})"


left = Node("left")
right = Node("right")

left.next = right
right.next = left

cycle = {left, right}


## Problem 15 — Copy a two-node cycle


In [50]:
cycle_copy = deepcopy(cycle)

copy_by_name = {
    node.name: node
    for node in cycle_copy
}

copied_left = copy_by_name["left"]
copied_right = copy_by_name["right"]


### Step 1 — Verify new objects


In [51]:
assert copied_left is not left
assert copied_right is not right


### Step 2 — Verify the copied cycle


In [52]:
print(copied_left.next is copied_right)
print(copied_right.next is copied_left)


True
True


### Solution 15

The cycle is preserved entirely within the copied graph.


In [53]:
assert copied_left.next is copied_right
assert copied_right.next is copied_left


# Part 16 — Custom deep-copy behavior


Generic deep copying is not always the desired ownership model.

A class may contain some references that should remain shared.


In [54]:
class Database:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Database({self.name!r})"


class Query:
    def __init__(self, sql, database, options):
        self.sql = sql
        self.database = database
        self.options = options

    def __deepcopy__(self, memo):
        if id(self) in memo:
            return memo[id(self)]

        clone = type(self).__new__(type(self))
        memo[id(self)] = clone

        clone.sql = self.sql
        clone.database = self.database
        clone.options = deepcopy(self.options, memo)

        return clone

    def __repr__(self):
        return f"Query({self.sql!r})" 


## Problem 16 — Copy query options but share the database


In [55]:
db = Database("analytics")

query = Query(
    "SELECT * FROM events",
    db,
    {
        "timeout": 5,
        "tags": ["daily"],
    },
)

queries = {query}
queries_copy = deepcopy(queries)

query_copy = next(iter(queries_copy))


### Step 1 — Check the query object


In [56]:
print("same query:", query_copy is query)


same query: False


### Step 2 — Check the database reference


In [57]:
print("same database:", query_copy.database is query.database)


same database: True


### Step 3 — Check mutable options


In [58]:
print(
    "same options dict:",
    query_copy.options is query.options,
)

print(
    "same tags list:",
    query_copy.options["tags"] is query.options["tags"],
)


same options dict: False
same tags list: False


### Solution 16

The custom `__deepcopy__` encodes the intended ownership model:

- new query;
- shared database;
- independent options.

The `memo` dictionary is essential for correct behavior with cycles and repeated references.


In [59]:
assert query_copy is not query
assert query_copy.database is query.database
assert query_copy.options is not query.options
assert query_copy.options["tags"] is not query.options["tags"]


# Part 17 — Explicit cloning as an alternative


Sometimes an explicit `clone()` method is easier to understand than generic deep-copy behavior.


In [60]:
class Worker:
    def __init__(self, worker_id, queue, preferences):
        self.worker_id = worker_id
        self.queue = queue
        self.preferences = preferences

    def clone(self):
        return Worker(
            worker_id=self.worker_id,
            queue=self.queue,
            preferences=deepcopy(self.preferences),
        )


shared_queue = {"name": "critical"}

worker = Worker(
    "W-1",
    shared_queue,
    {"retries": [1, 2, 3]},
)


## Problem 17 — Clone according to ownership rules


In [61]:
worker_clone = worker.clone()

print("same worker:", worker_clone is worker)
print("same queue:", worker_clone.queue is worker.queue)
print(
    "same preferences:",
    worker_clone.preferences is worker.preferences,
)
print(
    "same retry list:",
    worker_clone.preferences["retries"]
    is worker.preferences["retries"],
)


same worker: False
same queue: True
same preferences: False
same retry list: False


### Solution 17

The queue stays shared, while preferences become independent.

This intent is visible directly in the `clone()` implementation.


# Part 18 — `frozenset` protects membership


If callers should not be able to add or remove members, a `frozenset` may be a better API result than returning a mutable copy.


## Problem 18 — What does `frozenset` actually freeze?


In [62]:
members = {
    Customer("C-1", "Alice"),
    Customer("C-2", "Bob"),
}

snapshot = frozenset(members)

print(type(snapshot).__name__)


frozenset


### Step 1 — Try to modify membership


In [63]:
try:
    snapshot.add(Customer("C-3", "Charlie"))
except AttributeError as exc:
    print(type(exc).__name__, exc)


AttributeError 'frozenset' object has no attribute 'add'


### Step 2 — Mutate an element


In [64]:
alice = next(
    member
    for member in snapshot
    if member.customer_id == "C-1"
)

alice.display_name = "Alice Updated"

print(snapshot)


frozenset({Customer('C-2', display_name='Bob'), Customer('C-1', display_name='Alice Updated')})


### Solution 18

`frozenset` prevents membership mutation.

It does not recursively freeze mutable member objects.


# Part 19 — Defensive shallow copies at API boundaries


In [65]:
class Registry:
    def __init__(self, entries):
        self._entries = set(entries)

    def entries(self):
        return self._entries.copy()


entry = Customer("C-9", "Nina")
registry = Registry({entry})


## Problem 19 — What does the defensive copy protect?


In [66]:
external = registry.entries()

external.clear()

print("external size:", len(external))
print("registry size:", len(registry.entries()))


external size: 0
registry size: 1


### Step 2 — Does it protect member state?


In [67]:
entry.display_name = "Nina Changed"

inside = next(iter(registry.entries()))

print(inside.display_name)


Nina Changed


### Solution 19

The defensive shallow copy protects only the registry's internal set membership.

The member objects themselves are still shared.


# Part 20 — Immutable snapshot DTOs


Instead of copying entire mutable domain objects, an API can expose immutable value snapshots.


In [68]:
@dataclass(frozen=True)
class CustomerSnapshot:
    customer_id: str
    display_name: str


def make_customer_snapshot(customers):
    return frozenset(
        CustomerSnapshot(
            customer.customer_id,
            customer.display_name,
        )
        for customer in customers
    )


## Problem 20 — Snapshot values at a point in time


In [69]:
customers = {
    Customer("C-1", "Alice"),
    Customer("C-2", "Bob"),
}

snapshots = make_customer_snapshot(customers)

print(snapshots)


frozenset({CustomerSnapshot(customer_id='C-2', display_name='Bob'), CustomerSnapshot(customer_id='C-1', display_name='Alice')})


### Step 2 — Change an original object


In [70]:
alice = next(
    c for c in customers
    if c.customer_id == "C-1"
)

alice.display_name = "Alice New"

print("current object:", alice)
print("snapshot:", snapshots)


current object: Customer('C-1', display_name='Alice New')
snapshot: frozenset({CustomerSnapshot(customer_id='C-2', display_name='Bob'), CustomerSnapshot(customer_id='C-1', display_name='Alice')})


### Solution 20

The snapshot remains unchanged because it contains separate immutable value objects created from the original state.


# Part 21 — Set subclasses can have surprising copy behavior


Built-in container subclasses may carry extra metadata.

Do not assume inherited copying automatically preserves that metadata.


In [71]:
class TaggedSet(set):
    def __init__(self, iterable=(), tag=None):
        super().__init__(iterable)
        self.tag = tag


tagged = TaggedSet({1, 2, 3}, tag="important")


## Problem 21 — Inspect `.copy()` on the subclass


In [72]:
tagged_copy = tagged.copy()

print("original type:", type(tagged).__name__)
print("copy type:", type(tagged_copy).__name__)
print("copy tag:", getattr(tagged_copy, "tag", None))


original type: TaggedSet
copy type: set
copy tag: None


### Step 2 — Inspect `copy.copy()`


In [73]:
tagged_copy_module = copy(tagged)

print("copy.copy type:", type(tagged_copy_module).__name__)
print(
    "copy.copy tag:",
    getattr(tagged_copy_module, "tag", None),
)


copy.copy type: TaggedSet
copy.copy tag: important


### Solution 21

Subclass copy behavior should be tested explicitly whenever extra subclass state matters.

If copying is part of the class contract, implementing custom copy behavior can make expectations clear.


# Part 22 — Custom shallow copy for a set subclass


In [74]:
class SafeTaggedSet(set):
    def __init__(self, iterable=(), tag=None):
        super().__init__(iterable)
        self.tag = tag

    def __copy__(self):
        return type(self)(self, tag=self.tag)


special = SafeTaggedSet(
    {Note("A"), Note("B")},
    tag="study",
)


## Problem 22 — Preserve subclass metadata with `copy.copy()`


In [75]:
special_copy = copy(special)

print(type(special_copy).__name__)
print(special_copy.tag)

print(
    "same member ids:",
    element_ids(special_copy) == element_ids(special),
)


SafeTaggedSet
study
same member ids: True


### Solution 22

The custom `__copy__` preserves:

- subclass type;
- custom tag;
- shallow element sharing.


In [76]:
assert isinstance(special_copy, SafeTaggedSet)
assert special_copy is not special
assert special_copy.tag == "study"
assert element_ids(special_copy) == element_ids(special)


# Part 23 — Copying is not filtering


If the new set contains transformed or filtered values, the task is not a plain copy.

Use a comprehension or set operation that communicates the transformation.


## Problem 23 — Create a filtered independent set


In [77]:
numbers = {1, 2, 3, 4, 5, 6}

evens = {
    number
    for number in numbers
    if number % 2 == 0
}

print(evens)


{2, 4, 6}


### Solution 23

A comprehension is clearer than:

1. copying the original set;
2. mutating the copy repeatedly to remove unwanted values.


In [78]:
assert evens == {2, 4, 6}
assert numbers == {1, 2, 3, 4, 5, 6}


# Part 24 — Set algebra already returns new sets


Non-mutating operators create new sets:

```python
a | b
a & b
a - b
a ^ b
```

You do not need a preliminary copy just to preserve the inputs.


## Problem 24 — Combine permissions without modifying inputs


In [79]:
base_permissions = {"read", "write"}
temporary = {"admin"}

combined = base_permissions | temporary

print("base:", base_permissions)
print("temporary:", temporary)
print("combined:", combined)


base: {'write', 'read'}
temporary: {'admin'}
combined: {'write', 'admin', 'read'}


### Solution 24

Both source sets remain unchanged, and the union is a new set.


In [80]:
assert base_permissions == {"read", "write"}
assert temporary == {"admin"}
assert combined == {"read", "write", "admin"}


# Part 25 — Testing shallow-copy semantics


Printed output is educational, but good tests should directly assert the intended relationships.


## Problem 25 — Write a reusable shallow-copy assertion


In [81]:
def assert_shallow_set_copy(original, candidate):
    assert candidate is not original
    assert len(candidate) == len(original)
    assert element_ids(candidate) == element_ids(original)


### Step 1 — Test it


In [82]:
items = {
    Note("A"),
    Note("B"),
    Note("C"),
}

candidate = items.copy()

assert_shallow_set_copy(items, candidate)

print("shallow-copy checks passed")


shallow-copy checks passed


### Solution 25

The helper verifies exactly what shallow-copy semantics require at the direct set-member level.


# Part 26 — Testing deep separation


For a set of mutable objects, a basic deep-copy check is that no direct member identity is shared.

This does not prove every nested object is independent, so additional domain-specific checks may be required.


## Problem 26 — Build the first-level deep-copy assertion


In [83]:
def assert_no_direct_aliasing(original, candidate):
    assert candidate is not original
    assert len(candidate) == len(original)
    assert element_ids(original).isdisjoint(
        element_ids(candidate)
    )


In [84]:
items = {
    Course("A", ["x"]),
    Course("B", ["y"]),
}

candidate = deepcopy(items)

assert_no_direct_aliasing(items, candidate)

print("direct identity separation passed")


direct identity separation passed


### Step 2 — Verify nested lists as well


In [85]:
original_by_name = {
    item.name: item
    for item in items
}

candidate_by_name = {
    item.name: item
    for item in candidate
}

for name in original_by_name:
    assert (
        original_by_name[name].topics
        is not
        candidate_by_name[name].topics
    )

print("nested list separation passed")


nested list separation passed


### Solution 26

Deep-copy testing should follow the ownership requirements of the real object graph, not rely only on one generic equality check.


# Part 27 — Benchmark three shallow-copy styles


When performance matters, measure it.

Do not assume one shallow-copy spelling is always fastest across Python versions.


In [86]:
import timeit

sample = set(range(50_000))


## Problem 27 — Benchmark `.copy()`, `set(s)`, and unpacking


In [87]:
results = {
    "copy()": timeit.timeit(
        "sample.copy()",
        globals=globals(),
        number=100,
    ),
    "set(sample)": timeit.timeit(
        "set(sample)",
        globals=globals(),
        number=100,
    ),
    "{*sample}": timeit.timeit(
        "{*sample}",
        globals=globals(),
        number=100,
    ),
}

for name, seconds in sorted(
    results.items(),
    key=lambda pair: pair[1],
):
    print(f"{name:12} {seconds:.6f} s")


{*sample}    0.137161 s
set(sample)  0.156981 s
copy()       0.179925 s


### Solution 27

Use performance measurements for performance questions.

Use readability for ordinary application code. For a known set, `.copy()` normally communicates intent very directly.


# Part 28 — Design a safe mutable record


Requirements:

- a stable `record_id`;
- mutable metadata;
- safe use in sets;
- deep-copy independence.


In [88]:
class Record:
    def __init__(self, record_id, metadata):
        self.record_id = record_id
        self.metadata = dict(metadata)

    def __eq__(self, other):
        if not isinstance(other, Record):
            return NotImplemented
        return self.record_id == other.record_id

    def __hash__(self):
        return hash(self.record_id)

    def __repr__(self):
        return (
            f"Record({self.record_id!r}, "
            f"metadata={self.metadata!r})"
        )


## Problem 28 — Verify both hash safety and deep-copy independence


In [89]:
record = Record(
    "R-1",
    {"status": "new"},
)

records = {record}

hash_before = hash(record)

record.metadata["status"] = "done"

hash_after = hash(record)

print("hash stable:", hash_before == hash_after)
print("still in set:", record in records)


hash stable: True
still in set: True


### Step 2 — Deep-copy the set


In [90]:
records_copy = deepcopy(records)
copied_record = next(iter(records_copy))

print("same record object:", copied_record is record)
print(
    "same metadata dict:",
    copied_record.metadata is record.metadata,
)
print("sets value-equal:", records_copy == records)


same record object: False
same metadata dict: False
sets value-equal: True


### Solution 28

Mutable metadata is safe because hashing depends only on the stable ID.

The deep copy creates independent object state while logical equality remains based on `record_id`.


In [91]:
assert hash_before == hash_after
assert record in records
assert copied_record is not record
assert copied_record.metadata is not record.metadata
assert records_copy == records


# Part 29 — Remove, mutate, reinsert


Sometimes a legacy design has a mutable hash key.

If that key absolutely must change, one safer operational pattern is:

1. remove the object;
2. mutate the hash-relevant value;
3. insert it again.

This is still more fragile than a stable immutable key.


In [92]:
class RenameableKey:
    def __init__(self, key):
        self.key = key

    def __eq__(self, other):
        if not isinstance(other, RenameableKey):
            return NotImplemented
        return self.key == other.key

    def __hash__(self):
        return hash(self.key)

    def __repr__(self):
        return f"RenameableKey({self.key!r})" 


## Problem 29 — Change the key without corrupting the set


In [93]:
value = RenameableKey("old")
values = {value}

values.remove(value)

value.key = "new"

values.add(value)

print(values)


{RenameableKey('new')}


### Step 2 — Verify logical membership


In [94]:
assert value in values
assert RenameableKey("new") in values
assert RenameableKey("old") not in values


### Solution 29

The object was not inside the set while its hash changed.

This can work, but a stable hash key remains the better design when possible.


# Part 30 — Capstone tutorial: collaborative workspace


We will combine the major ideas into one realistic model.

Each member has:

- immutable logical `member_id`;
- mutable `display_name`;
- nested mutable preferences;
- a shared `Workspace`.

Required behavior:

1. display-name changes must not corrupt membership;
2. preference changes must not corrupt membership;
3. shallow copies should share member objects;
4. deep backup copies should create new member objects;
5. nested preferences should be independent in the backup;
6. the workspace should remain shared intentionally;
7. no test may depend on set order.


## Capstone Step 1 — Shared workspace


In [95]:
class Workspace:
    def __init__(self, workspace_id, name):
        self.workspace_id = workspace_id
        self.name = name

    def __repr__(self):
        return (
            f"Workspace({self.workspace_id!r}, "
            f"{self.name!r})"
        )


## Capstone Step 2 — Set-safe member


In [96]:
class Member:
    def __init__(
        self,
        member_id,
        display_name,
        preferences,
        workspace,
    ):
        self.member_id = member_id
        self.display_name = display_name
        self.preferences = deepcopy(preferences)
        self.workspace = workspace

    def __eq__(self, other):
        if not isinstance(other, Member):
            return NotImplemented
        return self.member_id == other.member_id

    def __hash__(self):
        return hash(self.member_id)

    def __deepcopy__(self, memo):
        if id(self) in memo:
            return memo[id(self)]

        clone = type(self).__new__(type(self))
        memo[id(self)] = clone

        clone.member_id = self.member_id
        clone.display_name = self.display_name
        clone.preferences = deepcopy(
            self.preferences,
            memo,
        )
        clone.workspace = self.workspace

        return clone

    def __repr__(self):
        return (
            f"Member({self.member_id!r}, "
            f"display_name={self.display_name!r})"
        )


## Capstone Step 3 — Create members


In [97]:
workspace = Workspace(
    "WS-1",
    "Advanced Python",
)

alice = Member(
    "M-1",
    "Alice",
    {
        "theme": "dark",
        "shortcuts": ["search"],
    },
    workspace,
)

bob = Member(
    "M-2",
    "Bob",
    {
        "theme": "light",
        "shortcuts": ["save"],
    },
    workspace,
)

members = {alice, bob}


## Capstone Step 4 — Mutate safe fields

The hash should remain stable.


In [98]:
alice_hash_before = hash(alice)

alice.display_name = "Alice Updated"
alice.preferences["theme"] = "system"
alice.preferences["shortcuts"].append("open")

alice_hash_after = hash(alice)

print(
    "hash stable:",
    alice_hash_before == alice_hash_after,
)

print(
    "still member:",
    alice in members,
)


hash stable: True
still member: True


In [99]:
assert alice_hash_before == alice_hash_after
assert alice in members


## Capstone Step 5 — Shallow membership copy

The outer set should be new, but the member objects should be shared.


In [100]:
membership_copy = members.copy()

print(
    "same outer set:",
    membership_copy is members,
)

print(
    "same member ids:",
    element_ids(membership_copy)
    == element_ids(members),
)


same outer set: False
same member ids: True


In [101]:
assert membership_copy is not members
assert element_ids(membership_copy) == element_ids(members)


## Capstone Step 6 — Deep backup copy

The backup should contain new member objects.


In [102]:
backup = deepcopy(members)

assert backup is not members
assert element_ids(backup).isdisjoint(
    element_ids(members)
)


## Capstone Step 7 — Match corresponding members by stable ID


In [103]:
original_by_id = {
    member.member_id: member
    for member in members
}

backup_by_id = {
    member.member_id: member
    for member in backup
}


## Capstone Step 8 — Verify the ownership model


In [104]:
for member_id in original_by_id:
    original_member = original_by_id[member_id]
    backup_member = backup_by_id[member_id]

    assert backup_member is not original_member

    assert (
        backup_member.preferences
        is not
        original_member.preferences
    )

    assert (
        backup_member.preferences["shortcuts"]
        is not
        original_member.preferences["shortcuts"]
    )

    assert (
        backup_member.workspace
        is
        original_member.workspace
    )

print("ownership checks passed")


ownership checks passed


## Capstone Step 9 — Prove backup independence


In [105]:
original_by_id["M-2"].preferences[
    "shortcuts"
].append("command-palette")

print(
    "original Bob:",
    original_by_id["M-2"].preferences,
)

print(
    "backup Bob:",
    backup_by_id["M-2"].preferences,
)


original Bob: {'theme': 'light', 'shortcuts': ['save', 'command-palette']}
backup Bob: {'theme': 'light', 'shortcuts': ['save']}


In [106]:
assert (
    backup_by_id["M-2"].preferences["shortcuts"]
    == ["save"]
)


## Capstone Solution

The final design separates three different ownership decisions:

### Set membership

A shallow `.copy()` creates a new membership container.

### Member state

A deep backup creates new member objects and independent nested preferences.

### Shared infrastructure

The workspace is intentionally preserved as a shared reference by custom `__deepcopy__`.

The stable `member_id` keeps equality and hashing safe even while display names and preferences change.


# Final Decision Guide

## Use assignment

```python
b = a
```

when you intentionally want two names for the same set.

## Use a shallow copy

```python
b = a.copy()
```

when membership should be independent but element objects should remain shared.

## Use `deepcopy`

```python
b = deepcopy(a)
```

when recursively independent mutable state is required and the object graph supports that semantic.

## Use explicit cloning

when some fields should be copied and others intentionally shared.

## Use `frozenset`

when callers need read-only membership.

## Use comprehensions or set algebra

when you are transforming, filtering, combining, or deriving values rather than merely copying.


# Final Hashing Checklist

Before placing a custom object in a set, ask:

1. Is the object hashable?
2. Does `__eq__` agree with `__hash__`?
3. If two objects compare equal, do they have equal hashes?
4. Can any hash-relevant field change?
5. Can equality and hashing use a stable immutable identity instead?
6. Would a frozen dataclass be a cleaner design?


# Final Copying Checklist

Before choosing shallow or deep copying, ask:

1. Does only set membership need independence?
2. Do direct member objects need independence?
3. Does nested state need independence?
4. Which references should intentionally remain shared?
5. Are cycles or shared nested references present?
6. Does the class need custom copy behavior?
7. Would an immutable snapshot be simpler?
8. Can explicit cloning better communicate ownership?


# Final Self-Test Problems

Try answering without running code:

1. Why does `.copy()` not protect you from mutations inside shared member objects?
2. Why can a mutable user-defined object be hashable?
3. What invariant is broken when an object's hash changes after insertion?
4. Why can two deep-copied sets contain different object identities but still be equal?
5. Why can two deep-copied sets fail equality even when their printed contents look similar?
6. What role does memoization play in `deepcopy()`?
7. What does `frozenset` make immutable?
8. Why might a set subclass need custom `__copy__` behavior?
9. Why is explicit cloning often preferable in domain models?
10. Why should tests never depend on set iteration order?


# Final Self-Test Solutions

1. A shallow copy duplicates only the outer set and reuses member references.
2. Hashability is about a stable hash/equality contract, not necessarily deep immutability.
3. The set can no longer reliably locate the object in the hash-table position determined at insertion time.
4. Their element classes may implement value equality using stable keys.
5. Their element classes may use identity equality instead of value equality.
6. Memoization prevents infinite recursion and preserves shared-reference relationships in copied graphs.
7. The membership structure of the `frozenset`; it does not recursively freeze mutable member objects.
8. Built-in copy behavior may not preserve subclass type or extra metadata as intended.
9. Explicit cloning states which references should be shared and which should be copied.
10. Sets are unordered, so order is not part of their semantic contract.
